**Import the useful packages** 

In [1]:
import numpy as np
import xarray as xr
import glob
import matplotlib.pyplot as plt
import os,sys
import pandas as pd

#import tensorflow as tf
from tensorflow import keras

sys.path.insert(1, '/bettik/ockendeh/SCRIPTS/simpleNN_basal_melt')

#from nn_functions.constants import *
#import nn_functions.diagnostic_functions as diag
#import nn_functions.data_formatting as dfmt
import nn_functions.postprocessing_functions as pp
#from nn_functions.constants import *

2025-01-17 15:03:27.754446: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-01-17 15:03:29.100448: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


**Set the parameters for this specific run**

In [128]:
mod_size = 'small'
TS_opt = 'extrap'
norm_method = 'std'
#exp_name = 'NEMO_grid_v0_slope_lat_lon'
exp_name = 'NEMO_grid_v0_slope_front'
#exp_name = 'test1'
exp_name = 'test_no_lat'
seed_nb = 1
#np.random.seed(seed_nb)
#tf.random.set_seed(seed_nb)

nemo_run = 'OPM026'
year = 2020
month = '09'

**Set the filepaths to the input files**

You will neeed:
- A keras model, and associated normalisation metrics 
- A dataset to apply the model to 

In [129]:
### use any model from CV over time
path_model = '/bettik/ockendeh/SCRIPTS/simpleNN_basal_melt/AIAI_data/NN_models/'
models = glob.glob(path_model + '*.keras')
filepath_model = path_model + 'model_nn_'+mod_size+'_'+exp_name+\
                                '_wholedataset_'+str(seed_nb).zfill(2)+'_TS'+TS_opt+'_norm'+norm_method+'.keras'

# Load the normalisation metrics (ideally these would be stored with the models?) 
filepath_AIAI_data = '/bettik/ockendeh/SCRIPTS/simpleNN_basal_melt/AIAI_data/'
filepath_norm_metrics = filepath_AIAI_data + 'normed_data_nonan_metrics_norm_wholedataset_v3.nc'

# Load in a data set to apply the model to 
application_data_fp = filepath_AIAI_data + 'Testing_data/nn_apply_' + nemo_run + '_' + str(year) + '_' + month + '.csv'

# Create a filepath to save the data to 
results_fp = filepath_AIAI_data + 'Testing_data/nn_applied_' + nemo_run + '_' + str(year) + '_' + month + '_' + exp_name + '.nc'

In [130]:
glob.glob(filepath_norm_metrics)

['/bettik/ockendeh/SCRIPTS/simpleNN_basal_melt/AIAI_data/normed_data_nonan_metrics_norm_wholedataset_v3.nc']

In [131]:
# Explore which other models and datasets are available in case this isn't the one you wanted
print('\033[1m' + 'You have requested the following model:' +'\033[0m')
print(os.path.split(filepath_model)[1])
print('\033[1m' + 'The other available models are:' + '\033[0m')
for i in range(len(models)):
    m_fp = os.path.split(models[i])[1]
    if m_fp != os.path.split(filepath_model)[1]:
        print(m_fp)
print('\033[1m' + 'If you wish to load one of these, please change the specified parameters' + '\033[0m')
print('')
print('\033[1m' + 'You have requested the following data set:' +'\033[0m')
print(application_data_fp)
print('\033[1m' + 'The other available data sets are:' + '\033[0m')
available_datasets = glob.glob(filepath_AIAI_data + 'Testing_data/nn_apply*.csv')
for i in range(len(available_datasets)):
    if available_datasets[i] != application_data_fp:
        print(available_datasets[i])
print('\033[1m' + 'If you wish to load one of these, please change the nemo_run, year or month' + '\033[0m')

You have requested the following model:
model_nn_small_test_no_lat_wholedataset_01_TSextrap_normstd.keras
The other available models are:
model_nn_small_test1_wholedataset_01_TSextrap_normstd.keras
model_nn_small_NEMO_grid_v0_slope_lat_lon_wholedataset_01_TSextrap_normstd.keras
model_nn_small_NEMO_grid_v0_slope_front_wholedataset_01_TSextrap_normstd.keras
If you wish to load one of these, please change the specified parameters

You have requested the following data set:
/bettik/ockendeh/SCRIPTS/simpleNN_basal_melt/AIAI_data/Testing_data/nn_apply_OPM026_2020_09.csv
The other available data sets are:
/bettik/ockendeh/SCRIPTS/simpleNN_basal_melt/AIAI_data/Testing_data/nn_apply_OPM026_2010_09.csv
/bettik/ockendeh/SCRIPTS/simpleNN_basal_melt/AIAI_data/Testing_data/nn_apply_OPM026_1980_09.csv
/bettik/ockendeh/SCRIPTS/simpleNN_basal_melt/AIAI_data/Testing_data/nn_apply_OPM026_2000_09.csv
/bettik/ockendeh/SCRIPTS/simpleNN_basal_melt/AIAI_data/Testing_data/nn_apply_OPM026_1990_09.csv
If you wis

In [132]:
# Load in the model 
model = keras.models.load_model(filepath_model)
print('You have loaded:')
print(filepath_model)

# Load in the normalisation metrics 
norm_metrics_file = xr.open_dataset(filepath_norm_metrics)
print(filepath_norm_metrics)
norm_metrics = norm_metrics_file.sel(norm_method=norm_method).drop('norm_method').to_dataframe()

# Load in the data to apply the model to
df_total = pd.read_csv(application_data_fp)
clean_df = df_total.to_xarray()
print(application_data_fp)

You have loaded:
/bettik/ockendeh/SCRIPTS/simpleNN_basal_melt/AIAI_data/NN_models/model_nn_small_test_no_lat_wholedataset_01_TSextrap_normstd.keras
/bettik/ockendeh/SCRIPTS/simpleNN_basal_melt/AIAI_data/normed_data_nonan_metrics_norm_wholedataset_v3.nc
/bettik/ockendeh/SCRIPTS/simpleNN_basal_melt/AIAI_data/Testing_data/nn_apply_OPM026_2020_09.csv


In [133]:
# And then normalise data 
val_norm = pp.normalise_vars(clean_df,
                            norm_metrics.loc['mean_vars'],
                            norm_metrics.loc['range_vars'])

# Should be the same list as used to make the data for the NN training, except without 'melt_m_ice_per_y' 
# I think
if exp_name == 'NEMO_grid_v0_slope_lat_lon':
    input_vars = ['distances_GL', 'distances_OO', 'distances_OC', 'temperature_prop', 'salinity_prop', \
                'corrected_isdraft', 'bathymetry', 'slope_is_lon', 'slope_is_lat', 'slope_ba_lon', \
                'slope_ba_lat', 'mean_T', 'mean_S', 'std_T', 'std_S']
elif exp_name == 'NEMO_grid_v0_slope_front':
    input_vars = ['distances_GL', 'distances_OO', 'distances_OC', 'temperature_prop', 'salinity_prop', \
                'corrected_isdraft', 'bathymetry', 'slope_is_across_front', 'slope_is_towards_front', \
                'slope_ba_across_front', 'slope_ba_towards_front', 'mean_T', 'mean_S', 'std_T', 'std_S']
elif exp_name == 'test1':
    input_vars = ['distances_GL', 'corrected_isdraft', 'temperature_prop', 'salinity_prop']
elif exp_name == 'test_no_lat':
    input_vars = ['corrected_isdraft', 'temperature_prop', 'salinity_prop']
else:
    print('I do not know the correct parameters for the exp_name you have requested')

x_val_norm = val_norm[input_vars]
y_val_norm = val_norm['melt_m_ice_per_y']

In [134]:
# Apply the model 
shape = x_val_norm.to_array().values.shape[1], x_val_norm.to_array().values.shape[0]
#y_out_norm = model.predict(x_val_norm.to_array().values.reshape(shape),verbose = 0)
y_out_norm = model.predict(x_val_norm.to_array().values.T,verbose = 0)

y_out_norm_xr = xr.DataArray(data=y_out_norm.squeeze()).rename({'dim_0': 'index'})
y_out_norm_xr = y_out_norm_xr.assign_coords({'index': x_val_norm.index})    

# denormalise the output
y_out = pp.denormalise_vars(y_out_norm_xr, 
                         norm_metrics['melt_m_ice_per_y'].loc['mean_vars'],
                         norm_metrics['melt_m_ice_per_y'].loc['range_vars'])

In [135]:
plot_figure = False
if plot_figure == True:
    import matplotlib.gridspec as gridspec
    fig = plt.figure(figsize=(14, 12))
    gs = gridspec.GridSpec(8,2, width_ratios=[3, 1], height_ratios = [0.5,1.5,0.2,1.5,0.5,0.2, 1,1])
    
    ax = [[],[],[],[]]
    ax[0] = fig.add_subplot(gs[0:2, 0])
    ax[1] = fig.add_subplot(gs[3:5, 0])
    ax[2] = fig.add_subplot(gs[6:8,0])
    ax[3] = fig.add_subplot(gs[1:3, 1])
    
    
    im = [[],[],[]]
    im[0] = ax[0].scatter(clean_df.lon, clean_df.lat, c = clean_df.melt_m_ice_per_y, cmap = 'coolwarm', s = 0.1)
    im[1] = ax[1].scatter(clean_df.lon, clean_df.lat, c = y_out, cmap = 'coolwarm', s = 0.1)
    im[2] = ax[2].scatter(clean_df.lon, clean_df.lat, c = y_out - clean_df.melt_m_ice_per_y, cmap = 'coolwarm', s = 0.1)
    titles = ['Reference melt', 'Predicted melt', 'Difference']
    #im[1] = ax[1].scatter(clean_df.lon, clean_df.lat, c = y_out, cmap = 'coolwarm')
    for i in range(3):
        im[i].set_clim(-0.0002,0.0002)
        ax[i].set_title(titles[i], fontsize =20)
        #ax[i].set_xlim(-65,-50)
        #ax[i].set_ylim(-83.6,-81.1)
    plt.colorbar(im[0], ax = ax[0:2], orientation = 'vertical', shrink = 0.5, pad = 0.02)
    ax[1].annotate('Variables: {}\nYear:        {}\nMonth:     {}'.format(exp_name, year, month), \
                   xy = (0,-75), xytext = (1.05,0.05), fontsize = 10, textcoords='axes fraction', zorder = 10);
    im[2].set_clim(-0.0001,0.0001)
    cbar = plt.colorbar(im[2], ax = ax[2], orientation = 'vertical', shrink = 1, pad = 0.02)
    cbar.set_ticks((-0.0001,0,0.0001))    
    
    ax[3].scatter(clean_df.melt_m_ice_per_y, y_out, s = 1)
    ax[3].set_xlabel('Reference melt')
    ax[3].set_ylabel('Predicted melt')
    ax[3].set_xticks((-0.001, 0))
    x_ample = np.arange(-0.0018,0.0005,0.0005)
    ax[3].plot(x_ample, x_ample, zorder = 0, color = 'grey', linewidth = 0.2)

SyntaxError: invalid syntax (2582386109.py, line 1)

In [136]:
y_out.to_netcdf(results_fp)
print('You saved:')
print(results_fp)

You saved:
/bettik/ockendeh/SCRIPTS/simpleNN_basal_melt/AIAI_data/Testing_data/nn_applied_OPM026_2020_09_test_no_lat.nc
